# Assignment 3: The "Multimodal Sentiment Engine" Challenge

**Total Marks:** 20 | **Deadline:** 7:00 PM, 22nd March, 2026 | 
**Submission:** A zip file of the folder containing this notebook, and the csv/image files you will create.


---

## Setup

Run the cell below **once** to install all required packages and download NLTK data.

In [1]:
!pip install -r requirements.txt -q

import nltk
for pkg in ["wordnet", "averaged_perceptron_tagger_eng", "punkt_tab", "omw-1.4"]:
    nltk.download(pkg, quiet=True)
print("Setup complete!")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup complete!


In [11]:
import os, re, json, time, random, warnings
from collections import Counter
from itertools import combinations

from dotenv import load_dotenv
load_dotenv()

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag

from gtts import gTTS
import librosa
import soundfile as sf
import os

warnings.filterwarnings("ignore")

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
LLM_MODEL = "google/gemini-2.0-flash-001"

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Sentiment-bearing words to preserve during augmentation
PRESERVE_WORDS = {
    "amazing", "terrible", "awful", "excellent", "wonderful", "horrible",
    "great", "bad", "good", "worst", "best", "love", "hate", "boring",
    "fantastic", "brilliant", "pathetic", "outstanding", "dreadful",
    "superb", "mediocre",
}

print("Imports loaded. API key present:", bool(OPENROUTER_API_KEY))

Imports loaded. API key present: True


## Task 1: Data Consolidation & Classical Augmentation (5 Marks)

**Steps:**
1. Load all three CSVs and merge them
2. Train a baseline Logistic Regression on `gold_standard_100.csv` (TF-IDF features)
3. Filter `llm_labels_150.csv` -- keep only reviews where baseline confidence ≥ 0.65 AND agrees with LLM label
4. Deduplicate by review text $\rightarrow$ save `consolidated_base.csv`
5. Identify minority class, apply 2 augmentation methods (Synonym Replacement, Back Translation)
6. Quality filter augmented samples (Jaccard similarity)
7. Save `augmented_classical.csv` and `class_distribution.png`

In [4]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
print(f"Gold: {len(gold)}, Weak: {len(weak)}, LLM: {len(llm)}")

def train_baseline_model(train_df, text_col="review", label_col="label"):
    """Returns (vectorizer, classifier) trained on the given dataframe."""
    vec = TfidfVectorizer(max_features=5000, stop_words="english")
    X = vec.fit_transform(train_df[text_col])
    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(X, train_df[label_col])
    return vec, clf

vec, clf = train_baseline_model(gold)

# 1c. Filter LLM labels by confidence
X_llm = vec.transform(llm["review"])

preds = clf.predict(X_llm)
probs = clf.predict_proba(X_llm).max(axis=1)

mask = (probs >= 0.65) & (preds == llm["label"])

filtered_llm = llm[mask].copy()

print("LLM original:", len(llm))
print("LLM filtered:", len(filtered_llm))


# 1d. Merge & deduplicate
combined = pd.concat([gold, weak, filtered_llm], ignore_index=True)
combined = combined.drop_duplicates(subset="review")
combined.to_csv("consolidated_base.csv", index=False)
print("Final dataset size:", len(combined))


# 1e. Class distribution analysis
class_counts = combined["label"].value_counts()

print("\nClass Distribution:")
print(class_counts)

minority_class = class_counts.idxmin()
print("\nMinority Class:", minority_class)

plt.figure()
class_counts.plot(kind="bar")
plt.xlabel("Sentiment")
plt.ylabel("Number of Samples")
plt.title("Class Distribution")
plt.tight_layout()

plt.savefig("class_distribution.png")
plt.show()

Gold: 100, Weak: 220, LLM: 150
LLM original: 150
LLM filtered: 29
Final dataset size: 320

Class Distribution:
label
Negative    148
Neutral     112
Positive     60
Name: count, dtype: int64

Minority Class: Positive


In [5]:
#  1f. Augmentation functions 

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            w = lemma.name().replace("_", " ").lower()
            if w != word:
                synonyms.add(w)
    return list(synonyms)


def synonym_replacement(text, replace_frac=0.15):
    tokens = word_tokenize(text)
    tagged = pos_tag(tokens)

    new_tokens = tokens.copy()

    # candidate words (exclude sentiment words)
    candidates = [
        i for i, (word, tag) in enumerate(tagged)
        if word.lower() not in PRESERVE_WORDS and word.isalpha()
    ]

    if not candidates:
        return text

    n_replace = max(1, int(len(tokens) * replace_frac))
    replace_indices = random.sample(candidates, min(n_replace, len(candidates)))

    for idx in replace_indices:
        word = tokens[idx].lower()
        syns = get_synonyms(word)
        if syns:
            new_tokens[idx] = random.choice(syns)

    return " ".join(new_tokens)

def back_translate(text, src="en", mid="hi"):
    """Translate English $\rightarrow$ Hindi $\rightarrow$ English using deep-translator GoogleTranslator."""
    from deep_translator import GoogleTranslator
    try:
        hindi = GoogleTranslator(source=src, target=mid).translate(text)
        time.sleep(0.5)
        english = GoogleTranslator(source=mid, target=src).translate(hindi)
        time.sleep(0.5)
        return english
    except Exception:
        return text

def quality_filter(original, augmented):
    set1 = set(original.lower().split())
    set2 = set(augmented.lower().split())

    if not set1 or not set2:
        return False

    intersection = len(set1 & set2)
    union = len(set1 | set2)

    jaccard = intersection / union

    return 0.30 <= jaccard <= 0.95

#  1g. Apply augmentation to minority class 
# TODO: For each minority-class sample, generate 2 augmented versions (one per method)
# TODO: Apply quality_filter, keep only passing samples
# TODO: Save augmented_classical.csv

data = pd.read_csv("consolidated_base.csv")
class_counts = data["label"].value_counts()
minority_class = class_counts.idxmin()
print("Minority class:", minority_class)
minority_samples = data[data["label"] == minority_class]
augmented_rows = []

for _, row in minority_samples.iterrows():
    text = row["review"]
    label = row["label"]

    aug1 = synonym_replacement(text)
    if quality_filter(text, aug1):
        augmented_rows.append({"review": aug1, "label": label})

    aug2 = back_translate(text)
    if quality_filter(text, aug2):
        augmented_rows.append({"review": aug2, "label": label})

augmented_df = pd.DataFrame(augmented_rows)

print("Generated augmented samples:", len(augmented_df))
augmented_df.to_csv("augmented_classical.csv", index=False)

Minority class: Positive
Generated augmented samples: 119


## Task 2: LLM-Based Synthetic Review Generation (5 Marks)

**Steps:**
1. Design a few-shot prompt with 3-4 gold-standard examples
2. Use OpenRouter API (via `openai` package) to generate 300 synthetic reviews in batches of 20
3. Calculate diversity metrics: Self-BLEU per class
4. Run sentiment consistency check with baseline model $\rightarrow$ flag mismatches
5. Save `llm_generated_300.csv`, `llm_generated_flagged.csv`, `prompt_template.txt`, `diversity_report.txt`

In [12]:
gold = pd.read_csv("gold_standard_100.csv")
weak = pd.read_csv("weak_labels_200.csv")
llm  = pd.read_csv("llm_labels_150.csv")
from openai import OpenAI

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=OPENROUTER_API_KEY)

#  2a. Design your few-shot prompt 
# TODO: Build a prompt string with 3-4 example reviews from gold standard
# Instruct the LLM to output JSON: [{"review": "...", "sentiment": "Positive", "movie": "..."}]
example_rows = []
for sentiment, group in gold.groupby("label"):
    # take up to 2 examples per sentiment
    samples = group.sample(min(2, len(group)), random_state=42)
    example_rows.append(samples)
examples = pd.concat(example_rows).reset_index(drop=True)

example_block = ""
for _, row in examples.iterrows():
    example_block += json.dumps({
        "review": row["review"],
        "sentiment": row["label"],
        "movie": "Example Movie"
    }) + "\n"

PROMPT_TEMPLATE = f"""
You are a movie review generator.
Generate realistic and diverse movie reviews.
Sentiment distribution must be:
- 10 Positive
- 7 Negative
- 3 Neutral
Requirements:
- Output strictly in JSON
- Format: [{{"review":"...", "sentiment":"Positive", "movie":"..."}}]
- Sentiment must be one of: Positive, Negative, Neutral
- Write natural human-like reviews.
Examples:
{example_block}
Now generate 20 new reviews.
"""
# Save prompt to file
with open("prompt_template.txt", "w", encoding="utf-8") as f:
    f.write(PROMPT_TEMPLATE)

#  2b. Generate reviews in batches 
# TODO: Loop to generate ~300 reviews in batches of 20
# Target distribution: ~150 Positive, ~100 Negative, ~50 Neutral
# Parse JSON response, handle errors
generated = []

TARGET_TOTAL = 300
BATCH_SIZE = 20

while len(generated) < TARGET_TOTAL:
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "user", "content": PROMPT_TEMPLATE}
        ],
        temperature=0.9  # Higher temperature → more diversity(creative)
    )
    content = response.choices[0].message.content
    try:
        content = content.strip()

        if content.startswith("```"):
            content = content.split("```")[1]   # remove first ```
            content = content.replace("json", "", 1).strip()
        
        data = json.loads(content)
        VALID_SENTIMENTS = {"Positive", "Negative", "Neutral"}
        
        for item in data:
            if "review" not in item or "sentiment" not in item:
                continue
            sentiment = str(item["sentiment"]).strip().capitalize()
            if sentiment not in VALID_SENTIMENTS:
                continue
            generated.append({
                "review": item["review"],
                "sentiment": sentiment,
                "movie": item.get("movie","Unknown")
            })
    except Exception as e:
        print("JSON parse error:", e)
    time.sleep(1)
    
generated = generated[:TARGET_TOTAL]
gen_df = pd.DataFrame(generated)

target_counts = {
    "Positive":150,
    "Negative":100,
    "Neutral":50
}

balanced_rows = []
for sentiment, target in target_counts.items():
    subset = gen_df[gen_df["sentiment"] == sentiment]
    balanced_rows.append(subset.sample(
        min(len(subset), target),
        random_state=42
    ))
gen_df = pd.concat(balanced_rows).reset_index(drop=True)


#  2c. Diversity metrics  ----> Lower BLEU = more diversity.
# TODO: Calculate Self-BLEU per sentiment class using nltk.translate.bleu_score
from nltk.translate.bleu_score import sentence_bleu

def compute_self_bleu(texts):
    scores = []
    for i, text in enumerate(texts):
        refs = [t.split() for j,t in enumerate(texts) if j!=i]
        hyp = text.split()
        score = sentence_bleu(refs, hyp)
        scores.append(score)
    return np.mean(scores)

diversity_report = {}
for sentiment in gen_df["sentiment"].unique():
    texts = gen_df[gen_df["sentiment"]==sentiment]["review"].tolist()
    diversity_report[sentiment] = compute_self_bleu(texts)

#  2d. Sentiment consistency check 
# TODO: Use baseline model (vec, clf) to predict sentiment of each generated review
# TODO: Flag mismatches, save llm_generated_flagged.csv
X_gen = vec.transform(gen_df["review"])
preds = clf.predict(X_gen)
gen_df["baseline_pred"] = preds
gen_df["match"] = gen_df["baseline_pred"] == gen_df["sentiment"]
flagged = gen_df[gen_df["match"] == False]
flagged.to_csv("llm_generated_flagged.csv", index=False)

#  2e. Save outputs 
# TODO: Save llm_generated_300.csv and diversity_report.txt
gen_df.to_csv("llm_generated_300.csv", index=False)
with open("diversity_report.txt","w") as f:
    for k,v in diversity_report.items():
        f.write(f"{k} Self-BLEU: {v}\n")
        

KeyboardInterrupt: 

## Task 3: Multilingual Sentiment Translation (4 Marks)

**Steps:**
1. Sample 100 reviews (40 Pos, 40 Neg, 20 Neu), prioritize shorter reviews
2. Translate English $\rightarrow$ Hindi using `deep-translator` (`GoogleTranslator`)
3. Back-translate Hindi $\rightarrow$ English, compute BLEU score (threshold ≥ 0.3)
4. Check sentiment preservation on back-translated text
5. Manually verify 5 random samples
6. Save `bilingual_reviews.csv` with `bleu_score` and `quality_flag` columns

In [7]:
from deep_translator import GoogleTranslator
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

#  3a. Strategic sampling 
# TODO: From consolidated_base, sample 100 reviews (40 Pos, 40 Neg, 20 Neu)
# Prioritize shorter reviews (sort by length, take shortest)
consolidated = pd.read_csv("consolidated_base.csv")

consolidated["length"] = consolidated["review"].apply(lambda x: len(str(x).split()))
samples = []
targets = {
    "Positive": 40,
    "Negative": 40,
    "Neutral": 20
}

for sentiment, count in targets.items():
    subset = consolidated[consolidated["label"] == sentiment]
    subset = subset.sort_values("length")  # prioritize shorter reviews
    samples.append(subset.head(count))
sample_df = pd.concat(samples).reset_index(drop=True)

#  3b. Translation pipeline 
# TODO: Translate each review English $\rightarrow$ Hindi using GoogleTranslator(source='en', target='hi')
# Add time.sleep(0.5) between calls to avoid rate limits

from deep_translator import GoogleTranslator
import time
hindi_reviews = []
for text in sample_df["review"]:
    try:
        hindi = GoogleTranslator(source='en', target='hi').translate(text)
    except:
        hindi = ""
    hindi_reviews.append(hindi)
    time.sleep(0.5)  # avoid rate limit
sample_df["hindi"] = hindi_reviews

#  3c. Back-translation & BLEU 
# TODO: Translate Hindi $\rightarrow$ English
# Compute sentence BLEU between original and back-translated
# quality_flag = "PASS" if BLEU >= 0.3, else "FAIL"

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

back_translated = []
bleu_scores = []
quality_flags = []

smooth = SmoothingFunction().method1

for original, hindi in zip(sample_df["review"], sample_df["hindi"]):
    try:
        back = GoogleTranslator(source='hi', target='en').translate(hindi)
    except:
        back = ""

    back_translated.append(back)
    reference = original.split()
    candidate = back.split()
    bleu = sentence_bleu([reference], candidate, smoothing_function=smooth)
    bleu_scores.append(bleu)
    if bleu >= 0.3:
        quality_flags.append("PASS")
    else:
        quality_flags.append("FAIL")
    time.sleep(0.5)

sample_df["back_translated"] = back_translated
sample_df["bleu_score"] = bleu_scores
sample_df["quality_flag"] = quality_flags

#  3d. Sentiment preservation check 
# TODO: Predict sentiment on back-translated text, compare with original label

X_back = vec.transform(sample_df["back_translated"])
preds = clf.predict(X_back)
sample_df["predicted_sentiment"] = preds
sample_df["sentiment_match"] = sample_df["predicted_sentiment"] == sample_df["label"]

#  3e. Manual verification 
# TODO: Print 5 random samples for inspection

print(sample_df.sample(5)[[
    "review",
    "hindi",
    "back_translated",
    "label",
    "predicted_sentiment",
    "bleu_score"
]])

#  3f. Save 
# TODO: Save bilingual_reviews.csv with columns:
# review, label, hindi, back_translated, bleu_score, quality_flag
sample_df.to_csv("bilingual_reviews.csv", index=False)

                                               review  \
83  Neither good nor bad, just exists. Acceptable,...   
53  A frustrating experience. Zero stars if I coul...   
70  I have mixed feelings about this one. Good for...   
45  The opening scene was frankly unbearable. Tota...   
44  A frustrating experience. Avoid this at all co...   

                                                hindi  \
83  न अच्छा, न बुरा, बस अस्तित्व में है। स्वीकार्य...   
53  एक निराशाजनक अनुभव. यदि मैं इसे दे सकूं तो शून...   
70  इसके बारे में मेरी मिश्रित भावनाएँ हैं। एक बार...   
45           शुरूआती दृश्य सचमुच असहनीय था। कुल कचरा.   
44          एक निराशाजनक अनुभव. इससे हर कीमत पर बचें.   

                                      back_translated     label  \
83  Neither good, nor bad, just exists. Acceptable...   Neutral   
53  A disappointing experience. Zero stars if I co...  Negative   
70  I have mixed feelings about it. Nice to see once.  Negative   
45  The initial scene was truly unbearable. To

## Task 4: Multimodal Audio Generation (4 Marks)

**Steps:**
1. Select 30 reviews (10 per class) of varying lengths
2. Use `gTTS` to generate audio (`tld="com"`), convert mp3$\rightarrow$wav via `librosa`+`soundfile`
3. Extract audio features with `librosa`: duration, spectral centroid, zero crossing rate, MFCCs
4. Use `openai-whisper` (tiny model) to transcribe audio back to text, compute WER
5. Save `audio_samples/` folder, `audio_features.csv`, `audio_validation.csv`

In [9]:
from gtts import gTTS
import librosa, soundfile as sf

#  4a. Select 30 reviews (10 per class) 
# TODO: Sample from consolidated_base, mix short and long reviews

data = pd.read_csv("consolidated_base.csv")
data["length"] = data["review"].apply(lambda x: len(str(x).split()))
samples = []
for sentiment in ["Positive","Negative","Neutral"]:
    subset = data[data["label"] == sentiment].sort_values("length")
    short = subset.head(5)
    long = subset.tail(5)
    samples.append(pd.concat([short,long]))
audio_df = pd.concat(samples).reset_index(drop=True)
print(len(audio_df))

#  4b. TTS generation 
# TODO: For each review, generate audio with gTTS (tld="com")
# Save as mp3, then load with librosa and re-save as .wav via soundfile


os.makedirs("audio_samples", exist_ok=True)

audio_paths = []
for i,row in audio_df.iterrows():
    text = row["review"]
    mp3_path = f"audio_samples/sample_{i}.mp3"
    wav_path = f"audio_samples/sample_{i}.wav"

    # Generate speech
    tts = gTTS(text=text, lang="en", tld="com")
    tts.save(mp3_path)

    # Convert mp3 → wav
    y, sr = librosa.load(mp3_path, sr=None)
    sf.write(wav_path, y, sr)
    audio_paths.append(wav_path)

audio_df["wav_path"] = audio_paths


#  4c. Audio feature extraction 
# TODO: For each wav, extract with librosa:
#   - duration (librosa.get_duration)
#   - spectral centroid (librosa.feature.spectral_centroid)
#   - zero crossing rate (librosa.feature.zero_crossing_rate)
#   - MFCCs (librosa.feature.mfcc, n_mfcc=13, take mean)
# Save audio_features.csv
features = []

for path in audio_df["wav_path"]:
    y, sr = librosa.load(path, sr=None)
    duration = librosa.get_duration(y=y, sr=sr)
    centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)

    row = {
        "wav_path": path,
        "duration": duration,
        "spectral_centroid": centroid,
        "zero_crossing_rate": zcr
    }
    for i,val in enumerate(mfcc_mean):
        row[f"mfcc_{i+1}"] = val
    features.append(row)

feature_df = pd.DataFrame(features)
feature_df.to_csv("audio_features.csv",index=False)

#  4d. Whisper round-trip validation 
import whisper 

_whisper_model = whisper.load_model("tiny")
transcriptions = []

for path in audio_df["wav_path"]:
    result = _whisper_model.transcribe(path)
    transcriptions.append(result["text"])
    
audio_df["transcription"] = transcriptions
_whisper_model = whisper.load_model("tiny")

# TODO: Transcribe each wav with Whisper
# Compute WER (word-level Levenshtein distance / reference word count)
# Flag samples with WER > 0.25
# Save audio_validation.csv

def wer(reference, hypothesis):
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    dp = [[0]*(len(hyp_words)+1) for _ in range(len(ref_words)+1)]
    for i in range(len(ref_words)+1):
        dp[i][0] = i
    for j in range(len(hyp_words)+1):
        dp[0][j] = j

    for i in range(1,len(ref_words)+1):
        for j in range(1,len(hyp_words)+1):

            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],   # deletion
                    dp[i][j-1],   # insertion
                    dp[i-1][j-1]  # substitution
                )
    return dp[-1][-1] / len(ref_words)

wer_scores = []
flags = []

for orig,trans in zip(audio_df["review"], audio_df["transcription"]):
    score = wer(orig.lower(), trans.lower())
    wer_scores.append(score)
    if score > 0.25:
        flags.append("FAIL")
    else:
        flags.append("PASS")

audio_df["wer"] = wer_scores
audio_df["validation_flag"] = flags
audio_df.to_csv("audio_validation.csv",index=False)

30


100%|█████████████████████████████████████| 72.1M/72.1M [00:03<00:00, 20.2MiB/s]


## Task 5: Final Dataset Assembly & Model Evaluation (2 Marks)

**Steps:**
1. Merge all datasets: `consolidated_base.csv` + `augmented_classical.csv` + `llm_generated_300.csv` (excluding flagged) + English text from `bilingual_reviews.csv`
2. Deduplicate $\rightarrow$ save `final_augmented_dataset.csv`
3. Use `BlackBoxEvaluator` from `evaluator.py` to compare baseline vs augmented accuracy

In [22]:
from evaluator import BlackBoxEvaluator

#  5a. Assemble final dataset 
# TODO: Load consolidated_base, augmented_classical, llm_generated_300, bilingual_reviews
# Exclude flagged LLM reviews
# Merge all, deduplicate on "review" column
# Save final_augmented_dataset.csv

base = pd.read_csv("consolidated_base.csv")
print("Base dataset:", len(base))
classical = pd.read_csv("augmented_classical.csv")
print("Classical augmentation:", len(classical))
llm = pd.read_csv("llm_generated_300.csv")
print("LLM generated:", len(llm))
flagged = pd.read_csv("llm_generated_flagged.csv")
print("Flagged LLM reviews:", len(flagged))
bilingual = pd.read_csv("bilingual_reviews.csv")
print("Bilingual dataset:", len(bilingual))

flagged = pd.read_csv("llm_generated_flagged.csv")
llm_clean = llm[~llm["review"].isin(flagged["review"])]
print("LLM after removing flagged:", len(llm_clean))

bilingual_en = bilingual[["review","label"]]
print("English reviews from bilingual:", len(bilingual_en))

final_augmented = pd.concat(
    [base, classical, llm_clean, bilingual_en],
    ignore_index=True
)
print("Merged dataset (before dedup):", len(final_augmented))
# If label is missing, fill it using sentiment
final_augmented["label"] = final_augmented["label"].fillna(final_augmented["sentiment"])
final_augmented = final_augmented.drop(columns=["sentiment"], errors="ignore")
final_augmented = final_augmented.drop_duplicates(subset="review")
final_augmented = final_augmented.dropna(subset=["label"])
final_augmented.to_csv("final_augmented_dataset.csv", index=False)
print("Final dataset (after dedup):", len(final_augmented))

#  5b. Black-Box Evaluation 
evaluator = BlackBoxEvaluator()
test_df = pd.read_csv("gold_standard_100.csv")

# Baseline evaluation (small dataset only)
# TODO: baseline_acc = evaluator.run_evaluation(consolidated_base, test_df)
baseline_acc = evaluator.run_evaluation(base, test_df)

# Augmented evaluation (full dataset)
# TODO: augmented_acc = evaluator.run_evaluation(final_augmented, test_df)
augmented_acc = evaluator.run_evaluation(final_augmented, test_df)

# Print comparison
# print(f"Baseline accuracy:  {baseline_acc:.2%}")
# print(f"Augmented accuracy: {augmented_acc:.2%}")
# print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")
print(f"Baseline accuracy:  {baseline_acc:.2%}")
print(f"Augmented accuracy: {augmented_acc:.2%}")
print(f"Improvement:        {augmented_acc - baseline_acc:+.2%}")

Base dataset: 320
Classical augmentation: 119
LLM generated: 294
Flagged LLM reviews: 139
Bilingual dataset: 100
LLM after removing flagged: 155
English reviews from bilingual: 100
Merged dataset (before dedup): 694
Final dataset (after dedup): 592
Initializing Black-Box Embedder...
Embedder loaded successfully.

--- Evaluating: Model ---
Training on 220 samples (excluded 100 test overlaps)...
Accuracy: 75.00%
Classification Report:
              precision    recall  f1-score   support

    Negative       0.53      0.84      0.65        25
     Neutral       0.86      0.66      0.75        47
    Positive       0.96      0.82      0.88        28

    accuracy                           0.75       100
   macro avg       0.78      0.77      0.76       100
weighted avg       0.80      0.75      0.76       100

----------------------------------------

--- Evaluating: Model ---
Training on 492 samples (excluded 100 test overlaps)...
Accuracy: 80.00%
Classification Report:
              prec